# GDino Score Threshold Analysis -- PLS (COCO Val)

Explores how the choice of `SCORE_THR` (the pre-filter on `gdino_score` applied
to the raw detections before any spreading) affects PLS's AP/AS, on top of the
existing `alpha` sweep -- with `k` fixed at 20. Unlike `budget`, which only
changes *which* detections in a fixed population count as "labeled", changing
`SCORE_THR` changes the population itself (more/fewer raw GroundingDINO
detections survive the filter), so every threshold needs its own re-filtered
dataset and its own k-NN graph, not just a different index sample into a
shared one.

Uses the same matched dataset / evaluation pipeline as
`baseline_comparison.py`. **Run this notebook in the
`label_spreading` conda environment** (pyamgx + faiss-gpu). Long-running (see
Section 2 for the combo count) -- plan on running it unattended (e.g. in
`tmux`).

In [1]:
import os
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from src import spreading_conventional, set_graph, _apply_object_conditioning
from evaluation import evaluate_result

pd.set_option("display.max_columns", None)

# ── Paths ─────────────────────────────────────────────────────────────────────────
DATASET_PATH = "../data/matched_dataset/coco_val_matched.parquet"
AMGX_CONFIG  = "./configs/FGMRES_AGGREGATION.json"
RESULTS_DIR  = "../analysis/experiments/gdino_score_analysis/grid_search"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Shared config ─────────────────────────────────────────────────────────────────────────
ONLY_OBJECTS = False
DATA_SPACE   = "gdino_embedding"   # 256-dim decoder hidden state
RUN_NAME     = "gdino_score_analysis"

COMPUTE_AP = True
COMPUTE_AS = True
SOFT_GT_PATH      = "../data/soft_datasets/soft_COCO_2017_val.json"
SOFT_LABEL_THRESH = 0.5

N_JOBS = max(1, (os.cpu_count() or 4) - 2)
print(f"N_JOBS = {N_JOBS}")

/home/klees/miniconda3/envs/label_spreading/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


N_JOBS = 190


## 1  Load the full, unfiltered dataset

Filtering by `gdino_score >= SCORE_THR` happens per grid point below (Section
2), not once up front, since `SCORE_THR` is itself one of the two swept
axes.

In [2]:
df_full = pd.read_parquet(DATASET_PATH)
print(f"Full dataset: {len(df_full):,} detections  |  "
      f"{df_full['gdino_match'].sum():,} matched to soft GT  ({df_full['gdino_match'].mean():.1%})")
print(f"gdino_score range: [{df_full['gdino_score'].min():.4f}, {df_full['gdino_score'].max():.4f}]")

Full dataset: 1,500,000 detections  |  49,478 matched to soft GT  (3.3%)
gdino_score range: [0.0028, 0.9588]


## 2  Grid Search -- AP / AS vs. score threshold x alpha (k fixed at 20)

For every `SCORE_THR` in the grid, re-filters the full dataset, then sweeps
`alpha` at each of three annotation budgets, evaluating every grid point with
both AP and AS (`evaluate_result(..., compute_ap=True, compute_as=True)`).
`k` is pinned to 20 throughout (matches the other sweep plots' single-`k`
default in `annotation_benchmark_results.ipynb`), so PLS's k-NN graph
structure differs only across `SCORE_THR` (population changes), not across
`alpha` or `budget` -- the graph is built once per `SCORE_THR` and
reused across `alpha` and `budget`.

Grids:
- **SCORE_THR**: `[0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.4,
  0.5]` -- 11 values.
- **alpha**: same 15-value grid as the main grid search
  (`baseline_comparison.py`).
- **budget**: `[0.01, 0.10, 1.00]` -- three representative points instead of
  the full 7-value list, to keep the added score-threshold axis tractable.

That's 11 x 15 x 3 = **495** `evaluate_result` calls -- each also pays its
own k-NN graph build (FAISS + AMGX), unlike a budget-only sweep at fixed
`SCORE_THR`. Expect a similar order of magnitude runtime to the main grid
search.

In [3]:
SCORE_THR_GRID = [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
PLS_ALPHA_GRID = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.975, 0.99, 0.995, 0.999]
PLS_K = 20
BUDGETS_TO_TEST = [0.01, 0.10, 1.00]

n_combos = len(SCORE_THR_GRID) * len(PLS_ALPHA_GRID) * len(BUDGETS_TO_TEST)
print(f"SCORE_THR: {len(SCORE_THR_GRID)} values -> {SCORE_THR_GRID}")
print(f"alpha:     {len(PLS_ALPHA_GRID)} values -> {PLS_ALPHA_GRID}")
print(f"budget:    {len(BUDGETS_TO_TEST)} values -> {BUDGETS_TO_TEST}")
print(f"k fixed at {PLS_K}")
print(f"Total combinations: {n_combos}")

SCORE_THR: 11 values -> [0.01, 0.02, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
alpha:     15 values -> [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.975, 0.99, 0.995, 0.999]
budget:    3 values -> [0.01, 0.1, 1.0]
k fixed at 20
Total combinations: 495


In [4]:
def filter_dataset(df_full_in, score_thr):
    '''Re-filters the full dataset at a given SCORE_THR and prepares the columns
    PLS needs (soft_label unpacked to an (N, 81) array, is_object as the TP/FP
    regression target) -- same preparation as baseline_comparison.py, parameterized by
    score_thr.'''
    dataset = df_full_in[df_full_in["gdino_score"] >= score_thr].copy().reset_index(drop=True)
    sl = np.array([v for v in dataset["soft_label"].values])
    dataset["soft_label"] = list(sl)
    dataset["is_object"] = dataset["gdino_match"].astype(float)
    return dataset


def sample_budget_indices(n_total, budget, seed=42):
    np.random.seed(seed)
    n = max(1, int(n_total * budget))
    return np.random.choice(n_total, size=n, replace=True)


def _evaluate(ds):
    return evaluate_result(ds, compute_ap=COMPUTE_AP, compute_as=COMPUTE_AS,
                            soft_gt_path=SOFT_GT_PATH, soft_label_thresh=SOFT_LABEL_THRESH)


def solve_and_package(S, alpha, dataset, classes, indices, run_tag, k, n_samples):
    '''One alpha's solve, reusing a precomputed graph S instead of rebuilding it --
    S depends only on (score_thr, k), never on alpha or budget, so building it once
    per SCORE_THR (see the main loop below) avoids rebuilding it for every
    (alpha, budget) pair.
    Same p_hat/score_est/trials math label_spreading_object_conventional does
    internally.'''
    F, F_object, _, _, _ = spreading_conventional(alpha, S, dataset, "soft_label", "is_object",
                                                    classes, AMGX_CONFIG, indices)
    row_sum = F.sum(axis=1)
    uniform = np.full(len(classes), 1.0 / len(classes))
    p_hat = np.divide(F, row_sum[:, None], out=np.tile(uniform, (len(dataset), 1)), where=row_sum[:, None] > 0)

    obj_row_sum = F_object.sum(axis=1)
    score_est = np.divide(F_object, obj_row_sum[:, None], out=np.full((len(dataset), 2), 0.5),
                           where=obj_row_sum[:, None] > 0)

    ds = dataset.copy()
    ds["p_hat"] = list(p_hat)
    ds["prediction"] = p_hat.argmax(axis=1)
    ds["score_est"] = list(score_est)
    _apply_object_conditioning(ds)
    ds.attrs["run_meta"] = dict(
        dataset_name=run_tag, only_objects=ONLY_OBJECTS, data_space=DATA_SPACE,
        label_column="label", prob_label_column="soft_label", n_data=len(ds), k=k, alpha=alpha,
        n_samples=n_samples, config=AMGX_CONFIG, elapsed_time=0.0, time_NN=0.0, time_graph=0.0,
        time_setup=0.0, time_solver=0.0, time_spread=0.0,
    )
    return ds


def run_alpha_grid(S, dataset, classes, score_thr, budget, indices_b, n_samples_b, n_jobs=N_JOBS):
    '''Fits+evaluates PLS at k=PLS_K across the full alpha grid, reusing the same
    precomputed graph S for every alpha: fitting is sequential (single AMGX/GPU
    context), each batch's evaluate_result calls run in parallel .'''
    rows = []
    for start in range(0, len(PLS_ALPHA_GRID), n_jobs):
        batch = PLS_ALPHA_GRID[start:start + n_jobs]
        fitted = [
            solve_and_package(S, alpha, dataset, classes, indices_b,
                               f"{RUN_NAME}_st{score_thr}_b{budget}_a{alpha:.4f}", PLS_K, n_samples_b)
            for alpha in batch
        ]
        results = Parallel(n_jobs=n_jobs)(delayed(_evaluate)(ds) for ds in fitted)
        for alpha, res in zip(batch, results):
            row = res.iloc[0].to_dict()
            row.update(method="PLS", k=PLS_K, alpha=alpha, score_thr=score_thr, budget=budget, n_samples=n_samples_b)
            rows.append(row)
        done = min(start + n_jobs, len(PLS_ALPHA_GRID))
        print(f"      alpha: {done}/{len(PLS_ALPHA_GRID)} done", flush=True)
    return rows

In [5]:
grid_csv = os.path.join(RESULTS_DIR, f"{RUN_NAME}_grid_search_results.csv")
grid_rows = []
done_score_thrs = set()
if os.path.exists(grid_csv):
    prior = pd.read_csv(grid_csv)
    grid_rows = prior.to_dict("records")
    done_score_thrs = set(prior["score_thr"].unique())
    print(f"Resuming from checkpoint: score_thr already done {sorted(done_score_thrs)}", flush=True)

for score_thr in SCORE_THR_GRID:
    if score_thr in done_score_thrs:
        print(f"\n===== SCORE_THR = {score_thr} ===== (skipping, already in checkpoint)", flush=True)
        continue

    t0 = time.time()
    print(f"\n===== SCORE_THR = {score_thr} =====", flush=True)
    dataset = filter_dataset(df_full, score_thr)
    classes = sorted(dataset["label"].dropna().unique())
    print(f"  {len(dataset):,} detections after filtering  |  {dataset['is_object'].sum():,.0f} matched", flush=True)

    tg0 = time.time()
    S, _, time_NN, time_graph = set_graph(PLS_K, dataset, DATA_SPACE, n_data=len(dataset))
    print(f"  graph built in {time.time() - tg0:.1f}s (NN={time_NN:.1f}s, graph={time_graph:.1f}s)", flush=True)

    for budget in BUDGETS_TO_TEST:
        indices_b = sample_budget_indices(len(dataset), budget)
        n_samples_b = len(indices_b)
        print(f"  -- budget = {budget:.0%}  ({n_samples_b:,} samples) --", flush=True)
        rows = run_alpha_grid(S, dataset, classes, score_thr, budget, indices_b, n_samples_b)
        grid_rows.extend(rows)

    print(f"  SCORE_THR={score_thr} done in {time.time() - t0:.0f}s", flush=True)

    # checkpoint after every complete score_thr
    pd.DataFrame(grid_rows).to_csv(grid_csv, index=False)

grid_results = pd.DataFrame(grid_rows)
grid_results.to_csv(grid_csv, index=False)
print(f"\nSaved grid search results -> {grid_csv}")
grid_results.head()

Resuming from checkpoint: score_thr already done [np.float64(0.01), np.float64(0.02)]



===== SCORE_THR = 0.01 ===== (skipping, already in checkpoint)



===== SCORE_THR = 0.02 ===== (skipping, already in checkpoint)



===== SCORE_THR = 0.03 =====


  942,905 detections after filtering  |  49,440 matched


...

  -- budget = 100%  (19,788 samples) --


      alpha: 15/15 done


  SCORE_THR=0.5 done in 259s



Saved grid search results -> ../analysis/experiments/gdino_score_analysis/grid_search/gdino_score_analysis_grid_search_results.csv


,dataset,n_data,dim_reduction_technique,dimension,only_objects,prob_labels,alpha,n_samples,k,RMSE,RMSE_mod,RMSE_match,RMSE_match_mod,Acc,Acc_match,Acc_mod,Acc_match_mod,TVD,TVD_match,TVD_mod,TVD_match_mod,MAE,MAE_match,MAE_mod,MAE_match_mod,runtime,time_NN,time_graph,time_setup,time_solver,time_spread,config,AP,AP50,AP75,AP_small,AP_medium,AP_large,AR_1,AR_10,AR_100,AR_small,AR_medium,AR_large,AP_sl,AP50_sl,AP75_sl,AP_small_sl,AP_medium_sl,AP_large_sl,AR_1_sl,AR_10_sl,AR_100_sl,AR_small_sl,AR_medium_sl,AR_large_sl,AS,AS_small,AS_medium,AS_large,AS_strict,AS_strict_small,AS_strict_medium,AS_strict_large,method,score_thr,budget
0,gdino_score_analysis_st0.01_b0.01_a0.0500,1404341,gdino,256,False,soft_label,0.05,14043,20,0.029311,2.327528e+10,0.077487,7.842675e+09,0.956762,0.403036,0.930008,0.417103,0.502473,0.558645,1.034141e+09,1.565707e+09,0.012407,0.013794,2.553434e+07,3.865944e+07,0.0,0.0,0.0,0.0,0.0,0.0,./configs/FGMRES_AGGREGATION.json,0.161871,0.248704,0.166676,0.076509,0.184576,0.270789,0.223111,0.519507,0.689262,0.464867,0.704277,0.882276,0.169267,0.260959,0.174523,0.080844,0.190737,0.283002,0.226958,0.525259,0.690239,0.470512,0.705280,0.882091,0.574646,0.395192,0.653299,0.840469,0.178608,0.080906,0.231604,0.338202,PLS,0.01,0.01
1,gdino_score_analysis_st0.01_b0.01_a0.1000,1404341,gdino,256,False,soft_label,0.10,14043,20,0.028933,5.148991e+09,0.077579,1.881392e+09,0.957774,0.395840,0.934759,0.413687,0.502477,0.561579,3.215685e+08,3.635476e+08,0.012407,0.013866,7.939963e+06,8.976485e+06,0.0,0.0,0.0,0.0,0.0,0.0,./configs/FGMRES_AGGREGATION.json,0.165220,0.253143,0.170238,0.071606,0.187478,0.275636,0.226282,0.523958,0.689917,0.469623,0.704690,0.882440,0.170680,0.261911,0.176372,0.074913,0.192486,0.290173,0.228796,0.528799,0.690247,0.470445,0.704554,0.882513,0.575222,0.395228,0.654275,0.842034,0.179379,0.079782,0.230036,0.346973,PLS,0.01,0.01
2,gdino_score_analysis_st0.01_b0.01_a0.2000,1404341,gdino,256,False,soft_label,0.20,14043,20,0.027910,9.771244e+06,0.074362,9.877088e+05,0.959420,0.396507,0.962471,0.410817,0.502096,0.557178,1.357800e+06,1.799137e+05,0.012397,0.013757,3.352593e+04,4.442313e+03,0.0,0.0,0.0,0.0,0.0,0.0,./configs/FGMRES_AGGREGATION.json,0.170220,0.260165,0.175985,0.075237,0.189918,0.282677,0.229627,0.530855,0.690550,0.470905,0.705039,0.882505,0.177178,0.272231,0.182936,0.080291,0.197025,0.292328,0.230843,0.532693,0.690873,0.470988,0.706685,0.882858,0.576227,0.395653,0.655663,0.843741,0.190669,0.089666,0.240887,0.355798,PLS,0.01,0.01
3,gdino_score_analysis_st0.01_b0.01_a0.3000,1404341,gdino,256,False,soft_label,0.30,14043,20,0.027529,1.250087e+06,0.074136,1.529789e-01,0.959857,0.386078,0.965659,0.409342,0.502305,0.564729,1.690506e+05,5.630947e-01,0.012403,0.013944,4.174089e+03,1.390357e-02,0.0,0.0,0.0,0.0,0.0,0.0,./configs/FGMRES_AGGREGATION.json,0.172733,0.263509,0.178801,0.076690,0.191830,0.285997,0.232590,0.535101,0.691126,0.472072,0.705824,0.882867,0.179962,0.274580,0.186363,0.079253,0.198917,0.303150,0.232573,0.531891,0.690251,0.469793,0.705541,0.882691,0.576170,0.395654,0.655193,0.843996,0.189916,0.087134,0.239911,0.357797,PLS,0.01,0.01
4,gdino_score_analysis_st0.01_b0.01_a0.4000,1404341,gdino,256,False,soft_label,0.40,14043,20,0.027290,4.143089e+03,0.073657,7.295594e-02,0.960624,0.386462,0.966471,0.407705,0.502153,0.561104,5.553032e+02,5.454971e-01,0.012399,0.013854,1.371119e+01,1.346907e-02,0.0,0.0,0.0,0.0,0.0,0.0,./configs/FGMRES_AGGREGATION.json,0.175294,0.267144,0.181574,0.077613,0.193993,0.289230,0.235342,0.539110,0.691350,0.472392,0.705949,0.882820,0.183749,0.281653,0.189957,0.081875,0.204134,0.304076,0.235029,0.538850,0.690670,0.471409,0.704435,0.882913,0.576338,0.395914,0.655593,0.843782,0.194944,0.093080,0.244448,0.360962,PLS,0.01,0.01


## 3  Quick sanity check -- best AS vs. score threshold

The paper figures (`PLS_gdino_score_heatmap_*.pdf`) are made from this CSV in
`analysis/annotation_benchmark_results.ipynb`; this is just a quick look at the sweep.

In [ ]:
import matplotlib.pyplot as plt

best_as = (
    grid_results.sort_values("AS_strict", ascending=False)
    .groupby(["score_thr", "budget"], as_index=False).first()[["score_thr", "budget", "AS_strict", "alpha"]]
)

fig, ax = plt.subplots(figsize=(7, 5))
for budget in BUDGETS_TO_TEST:
    sub = best_as[best_as["budget"] == budget].sort_values("score_thr")
    ax.plot(sub["score_thr"], sub["AS_strict"], marker="o", label=f"{budget:.0%} budget")
ax.set_xscale("log")
ax.set_xlabel("Score threshold (gdino_score pre-filter)")
ax.set_ylabel("Best AS (over alpha, k=20)")
ax.set_title("PLS: best AS vs. score threshold")
ax.grid(True, ls=":")
ax.legend()
plt.tight_layout()
plt.show()